In [2]:
!pip install -U "transformers>=4.38.0" datasets accelerate evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 156.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 51.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset

from google.colab import files

# Transformers 관련
from transformers import (
    AutoTokenizer, AutoModel,
    TrainingArguments, Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    TrainerCallback
)

# 성능 평가 관련
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report, f1_score

# --------------------------------------------
# 0. 기본 설정
# --------------------------------------------
SEED = 42
MODEL_NAME = "klue/roberta-base"
SAVE_PATH = "/content/drive/MyDrive/10_Models/3-1_KLUE-MoE/"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 장치:", device)

# --------------------------------------------
# 1. Google Drive 마운트
# --------------------------------------------
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

# 저장 경로 설정
SAVE_PATH = "/content/drive/MyDrive/10_Models/3-1_KLUE-MoE/"
os.makedirs(SAVE_PATH, exist_ok=True)
os.makedirs(os.path.join(SAVE_PATH, "logs"), exist_ok=True)

# CUDA 사용 여부 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# -----------------------------------
# 1) 데이터셋 로드
# -----------------------------------
print("CSV 또는 XLSX 파일을 업로드하세요.")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# 파일 확장자 확인 후 데이터 로드
if filename.endswith('.csv'):
    data = pd.read_csv(filename)
elif filename.endswith('.xlsx'):
    data = pd.read_excel(filename)
else:
    raise ValueError("지원되지 않는 파일 형식입니다. CSV 또는 XLSX 파일을 업로드하세요.")

# -----------------------------------
# 2) 데이터 전처리
# -----------------------------------
attribute_columns = [
    'INFOSRC', 'INFOSRC_CODE_2', 'ART_DATE', 'ART_PROVIDER', 'ART_CATEGORY1','ART_CATEGORY2', 'ART_CATEGORY3',
    'ART_TAG_1', 'ART_TAG_2', 'ART_TAG_3', 'SNT_TAG_1', 'SNT_TAG_2','SNT_TAG_3', 'ART_HEADLINE', 'ART_BYLINE'
]

required_columns = ["STN_CONTENT", "OPN"] + attribute_columns

missing_columns = [col for col in required_columns if col not in data.columns]

if missing_columns:
    raise ValueError(f"다음 필수 컬럼이 없습니다: {missing_columns}")

# --------------------------------------------
# 4. 결측값 처리 및 속성 문자열 생성
# --------------------------------------------
data = data.copy()

data["STN_CONTENT"] = data["STN_CONTENT"].fillna("").astype(str)
data["OPN"] = data["OPN"].fillna("").astype(str)

for col in attribute_columns:
    data[col] = data[col].fillna("").astype(str)

# 빈 본문 또는 빈 라벨 제거
data = data[
    (data["STN_CONTENT"].str.strip() != "") &
    (data["OPN"].str.strip() != "")
].reset_index(drop=True)

if len(data) == 0:
    raise ValueError("유효한 학습 데이터가 없습니다. STN_CONTENT 또는 OPN 컬럼을 확인하십시오.")

data["attributes_text"] = data[attribute_columns].agg(" ".join, axis=1)

# --------------------------------------------
# 5. 라벨 인코딩
# --------------------------------------------
label_encoder = LabelEncoder()
data["label"] = label_encoder.fit_transform(data["OPN"])

label_names = list(label_encoder.classes_)
num_labels = len(label_names)

print("라벨 목록:", label_names)
print("라벨 수:", num_labels)
print("라벨 분포:")
print(data["OPN"].value_counts())

if num_labels < 2:
    raise ValueError("라벨이 2개 미만입니다. 분류 모델 학습이 불가능합니다.")

# --------------------------------------------
# 6. 데이터 셔플 및 Hugging Face Dataset 변환
# --------------------------------------------
data = data.sample(frac=1, random_state=SEED).reset_index(drop=True)

hf_dataset = Dataset.from_pandas(
    data[["STN_CONTENT", "attributes_text", "label"]],
    preserve_index=False
)

# 먼저 80:20(train+eval : test)으로 분할 후,
# 남은 80%를 다시 75:25(train : eval)으로 분할
# => 최종 60:20:20(train : eval : test)

split_data = hf_dataset.train_test_split(test_size=0.2, seed=SEED)
test_dataset_hf = split_data['test']         # 20%
remaining_dataset = split_data['train']      # 80%

split_data2  = remaining_dataset.train_test_split(test_size=0.25, seed=42)  # 80% 중 25% = 20% (전체)
train_dataset_hf = split_data2['train']      # 최종 60%
eval_dataset_hf  = split_data2['test']       # 최종 20%

print(f"Train dataset size: {len(train_dataset_hf)}")
print(f"Eval dataset size:  {len(eval_dataset_hf)}")
print(f"Test dataset size:  {len(test_dataset_hf)}")

# -----------------------------------
# 7. 토크나이저
# -----------------------------------
MODEL_NAME = "klue/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(example):
    encoding1 = tokenizer(example['attributes_text'], padding='max_length', truncation=True, max_length=256)
    encoding2 = tokenizer(example['STN_CONTENT'], padding='max_length', truncation=True, max_length=256)
    return {
        'input_ids1': encoding1['input_ids'],
        'attention_mask1': encoding1['attention_mask'],
        'input_ids2': encoding2['input_ids'],
        'attention_mask2': encoding2['attention_mask'],
        'label': example['label']
    }
# -----------------------------------
# 8. Dataset 변환
# -----------------------------------
train_dataset = train_dataset_hf.map(tokenize_function, batched=True)
eval_dataset = eval_dataset_hf.map(tokenize_function, batched=True)
test_dataset = test_dataset_hf.map(tokenize_function, batched=True)

# 원문 컬럼 제거
columns_to_remove = ["STN_CONTENT", "attributes_text"]
train_dataset = train_dataset.remove_columns(columns_to_remove)
eval_dataset = eval_dataset.remove_columns(columns_to_remove)
test_dataset = test_dataset.remove_columns(columns_to_remove)

# -----------------------------------
# 6) KLUE-RoBERTa 활용 MoE 모델 정의
# -----------------------------------
class MoEBertModel(nn.Module):
    def __init__(self, num_labels):
        super(MoEBertModel, self).__init__()
        self.num_labels = num_labels
        self.hidden_size = 768

        # 전문가 1 (속성 문자열 입력)
        self.bert_expert1 = AutoModel.from_pretrained('klue/roberta-base')

        # 전문가 2 (인용문 입력)
        self.bert_expert2 = AutoModel.from_pretrained('klue/roberta-base')

        self.classifier1 = nn.Linear(self.hidden_size, num_labels)
        self.classifier2 = nn.Linear(self.hidden_size, num_labels)

        # 초기 레이어 일부 고정 (예: 첫 6개 레이어)
        for param in self.bert_expert1.encoder.layer[:6].parameters():
            param.requires_grad = False
        for param in self.bert_expert2.encoder.layer[:6].parameters():
            param.requires_grad = False

        # 게이팅 네트워크 (두 전문가의 pooled_output을 어떻게 섞을지 결정)
        self.gating_network = nn.Sequential(
            nn.Linear(self.hidden_size * 2, self.hidden_size),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(self.hidden_size, 2),
            nn.Softmax(dim=1))

        # 드롭아웃 + 최종 분류기
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.hidden_size, self.num_labels)

    def forward(self, input_ids1, attention_mask1, input_ids2, attention_mask2, labels=None):

        # 전문가 1
        encoding1 = self.bert_expert1(input_ids=input_ids1, attention_mask=attention_mask1).pooler_output

        # 전문가 2
        encoding2 = self.bert_expert2(input_ids=input_ids2, attention_mask=attention_mask2).pooler_output

        # 게이팅 네트워크: 2개 전문가 pooled_output을 concat하여 가중치 결정
        logits1 = self.classifier1(encoding1)
        logits2 = self.classifier2(encoding2)

        combined = torch.cat((encoding1, encoding2), dim=1)  # [batch_size, hidden_size * 2]
        gating_weights = self.gating_network(combined)  # [batch_size, 2]

        # 전문가 출력 스택
        expert_outputs = torch.stack([logits1, logits2], dim=1)  # [batch_size, 2, hidden_size]

        # 전문가 출력을 게이팅 가중치에 따라 가중 합
        gated_output = torch.einsum('bi,bih->bh', gating_weights, expert_outputs)  # [batch_size, hidden_size]

        # 손실 계산
        loss = None
        if labels is not None:
            criterion = nn.CrossEntropyLoss()
            loss = criterion(gated_output, labels)

        if loss is not None:
            return {
                "loss": loss,
                "logits": gated_output
            }

        return {
            "logits": gated_output
        }

# 모델 초기화 (Trainer보다 먼저 실행해야 함)
model = MoEBertModel(num_labels=num_labels).to(device)
print("모델이 정상적으로 생성되었습니다.")

# -----------------------------------
# 9. 평가 지표 함수
# -----------------------------------
def compute_metrics(eval_pred):
    labels = eval_pred.label_ids
    preds = np.argmax(eval_pred.predictions, axis=1)

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
    return {'accuracy': acc,'macro_f1': f1,'macro_precision': precision,'macro_recall': recall}

# -----------------------------------
# 9) Trainer 설정
# -----------------------------------
training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/10_Models/3-1_KLUE-MoE',
    num_train_epochs=100,
    per_device_train_batch_size=64, # 배치 크기 증가
    per_device_eval_batch_size=64,
    report_to="none",  # wandb 자동 실행 방지
    learning_rate=1e-5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),  # Mixed Precision 활성화 (속도 향상)

    # epoch 단위로 로그/평가/체크포인트 저장
    logging_strategy='epoch',
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,

    # 로그 저장 디렉토리
    logging_dir='/content/drive/MyDrive/10_Models/3-1_KLUE-MoE/logs',
    remove_unused_columns=False
)

# 조기 종료 콜백
early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=5,
    early_stopping_threshold=0.001
)

# -----------------------------------
# 10) 커스텀 데이터 콜레이터 사용
# -----------------------------------
def custom_data_collator(batch):
    return {
        'input_ids1': torch.stack([torch.tensor(item['input_ids1']) for item in batch]),
        'attention_mask1': torch.stack([torch.tensor(item['attention_mask1']) for item in batch]),
        'input_ids2': torch.stack([torch.tensor(item['input_ids2']) for item in batch]),
        'attention_mask2': torch.stack([torch.tensor(item['attention_mask2']) for item in batch]),
        'labels': torch.tensor([item['label'] for item in batch], dtype=torch.long),
    }

# -----------------------------------
# 11) 콜백 정의: 매 epoch 끝 Train 세트 평가
# -----------------------------------
class TrainDatasetMetricsCallback(TrainerCallback):
    """
    매 epoch가 끝날 때 Train 데이터셋을 사용하여
    추가적인 지표(accuracy, loss 등)를 계산하는 콜백 예시
    """
    def __init__(self, train_dataset):
        super().__init__()
        self.train_dataset = train_dataset
        self.trainer_ref = None

    def on_train_begin(self, args, state, control, **kwargs):
        # 학습 시작 시점에 trainer 객체를 저장해둠
        if "trainer" in kwargs:
            self.trainer_ref = kwargs["trainer"]

    def on_epoch_end(self, args, state, control, **kwargs):
        # 매 epoch 종료 시점에 train_dataset 평가
        if self.trainer_ref is not None:
            train_metrics = self.trainer_ref.evaluate(
                eval_dataset=self.train_dataset,
                metric_key_prefix="train"
            )
            train_acc  = train_metrics.get("train_accuracy", None)
            train_loss = train_metrics.get("train_loss", None)
            print(f"[Epoch {int(state.epoch)}] Train Accuracy: {train_acc:.4f}, Train Loss: {train_loss:.4f}")
        return control

# 콜백 인스턴스 생성
train_metrics_callback = TrainDatasetMetricsCallback(train_dataset)

# -----------------------------------
# 12) Trainer 객체 생성
# -----------------------------------
# Trainer에서 기존 `data_collator` 대신 `custom_data_collator` 사용
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=custom_data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,   # 검증 세트 지표 계산
    callbacks=[early_stopping_callback, train_metrics_callback]
)

# -----------------------------------
# 13) 모델 학습
# -----------------------------------
trainer.train()

# -----------------------------------
# 14) 평가 함수 정의
# -----------------------------------
def evaluate_and_report(trainer, dataset, dataset_name, label_names):
    predictions = trainer.predict(dataset)
    preds = predictions.predictions.argmax(-1)
    labels = predictions.label_ids

    acc = accuracy_score(labels, preds)
    f1_w = f1_score(labels, preds, average='weighted')
    print(f"\n===== [{dataset_name}] =====")
    print(f"{dataset_name} Accuracy: {acc:.4f}")
    print(f"{dataset_name} Weighted-F1: {f1_w:.4f}")

    # label_names 파라미터를 통해 레이블 순서 지정 가능
    print(classification_report(labels, preds, target_names=label_names))

# label_encoder.classes_에 따른 실제 라벨명 (예: ['매도','중립','매수'] 등)
label_names = list(label_encoder.classes_)
print("Label names:", label_names)

# -----------------------------------
# 15) 최종 평가 (Train/Val/Test)
# -----------------------------------
# (A) Train 세트 평가
evaluate_and_report(trainer, train_dataset, "Train Set", label_names)

# (B) Validation 세트 평가
eval_results = trainer.evaluate()  # eval_dataset=eval_dataset
print(f"\n[Validation Set] eval_loss: {eval_results['eval_loss']:.4f}")
evaluate_and_report(trainer, eval_dataset, "Validation Set", label_names)

# (C) Test 세트 평가
evaluate_and_report(trainer, test_dataset, "Test Set", label_names)

# -----------------------------------
# 16) 모델 및 가중치 저장
# -----------------------------------
torch.save(model, '/content/drive/MyDrive/10_Models/3-1_KLUE-MoE/best_model.pt')
torch.save(model.state_dict(), '/content/drive/MyDrive/10_Models/3-1_KLUE-MoE/best_model_weights.pt')
print("모델, 가중치를 저장했습니다")


사용 장치: cuda
Mounted at /content/drive
CSV 또는 XLSX 파일을 업로드하세요.


Saving labelled_OPN_98337_202401071715_attri&quote.csv to labelled_OPN_98337_202401071715_attri&quote.csv
라벨 목록: ['B', 'N', 'S']
라벨 수: 3
라벨 분포:
OPN
N    32779
S    32779
B    32779
Name: count, dtype: int64
Train dataset size: 59001
Eval dataset size:  19668
Test dataset size:  19668


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

...


===== [Train Set] =====
Train Set Accuracy: 0.9141
Train Set Weighted-F1: 0.9140
              precision    recall  f1-score   support

           B       0.92      0.94      0.93     19753
           N       0.90      0.91      0.90     19658
           S       0.92      0.90      0.91     19590

    accuracy                           0.91     59001
   macro avg       0.91      0.91      0.91     59001
weighted avg       0.91      0.91      0.91     59001



Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Macro Precision,Macro Recall
0.128869,0.291457,8,0.886567,0.886503,0.886565,0.886509



[Validation Set] eval_loss: 0.2915



===== [Validation Set] =====
Validation Set Accuracy: 0.8866
Validation Set Weighted-F1: 0.8865
              precision    recall  f1-score   support

           B       0.90      0.91      0.90      6591
           N       0.87      0.87      0.87      6539
           S       0.89      0.88      0.89      6538

    accuracy                           0.89     19668
   macro avg       0.89      0.89      0.89     19668
weighted avg       0.89      0.89      0.89     19668




===== [Test Set] =====
Test Set Accuracy: 0.8834
Test Set Weighted-F1: 0.8834
              precision    recall  f1-score   support

           B       0.89      0.91      0.90      6435
           N       0.86      0.87      0.87      6582
           S       0.89      0.87      0.88      6651

    accuracy                           0.88     19668
   macro avg       0.88      0.88      0.88     19668
weighted avg       0.88      0.88      0.88     19668

모델, 가중치를 저장했습니다
